# 🦺 SiteSafe Vision: MLOps Scripts & System Demonstration Notebook
### Interactive Verification, Performance Benchmarking, Reproducibility Audit & Live Inference

This notebook consolidates all utility and MLOps audit scripts from the `scripts/` directory into an interactive presentation layer for demonstrations, audits, and pipeline inspection.

---

## 1. Project Root & Path Initialization

In [ ]:
import sys
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image

# Ensure project root is in sys.path
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project Root: {PROJECT_ROOT}")

## 2. Environment & System Signature (`scripts/environment_report.py`)
Captures OS, CPU, GPU, CUDA, PyTorch, Torchvision, Git commit, and DVC versions.

In [ ]:
from scripts.environment_report import generate_environment_report

env_report = generate_environment_report(PROJECT_ROOT)

print("=" * 60)
print("  ENVIRONMENT SIGNATURE REPORT")
print("=" * 60)
print(f"OS: {env_report['system']['os']} {env_report['system']['os_release']}")
print(f"Python Version: {env_report['system']['python_version'].split()[0]}")
print(f"PyTorch Version: {env_report['dependencies']['torch']}")
print(f"Torchvision Version: {env_report['dependencies']['torchvision']}")
print(f"CUDA Available: {env_report['hardware']['cuda_available']}")
print(f"Git Commit: {env_report['version_control']['git_commit']}")
print(f"DVC Version: {env_report['version_control']['dvc_version']}")
print(f"Config SHA-256: {env_report['governance_hashes']['config_hash_sha256'][:16]}...")

## 3. Artifact Integrity & Provenance Lineage (`scripts/verify_artifacts.py`)
Computes SHA-256 checksums and traces the complete model lineage graph.

In [ ]:
from scripts.verify_artifacts import generate_checksums_and_lineage

# Run verification and generate lineage report
generate_checksums_and_lineage(PROJECT_ROOT)

lineage_path = PROJECT_ROOT / "reports" / "lineage.json"
with open(lineage_path, "r", encoding="utf-8") as f:
    lineage_data = json.load(f)

print("\n--- PROVENANCE LINEAGE TRACE ---")
for idx, step in enumerate(lineage_data["lineage_trace"], 1):
    print(f"{idx}. {step}")

print(f"\nProduction Model SHA-256: {lineage_data['production_model_sha256']}")

## 4. Performance Benchmarking (`scripts/benchmark_performance.py`)
Evaluates inference latency, parameter counts, and throughput across ResNet50 vs. MobileNetV3-Large.

In [ ]:
from scripts.benchmark_performance import run_full_benchmark

bench_results = run_full_benchmark()

mobilenet_lat = bench_results["architectures"]["mobilenet_v3_large"]["single_image_latency_ms"]
resnet_lat = bench_results["architectures"]["resnet50"]["single_image_latency_ms"]

bench_df = pd.DataFrame([
    {
        "Model Architecture": "MobileNetV3-Large (Champion)",
        "Parameters": f"{bench_results['architectures']['mobilenet_v3_large']['total_parameters'] / 1e6:.1f} M",
        "File Size (MB)": bench_results["architectures"]["mobilenet_v3_large"]["model_file_size_mb"],
        "Mean Latency (ms)": mobilenet_lat["mean"],
        "P95 Latency (ms)": mobilenet_lat["p95"],
    },
    {
        "Model Architecture": "ResNet50 (Baseline)",
        "Parameters": f"{bench_results['architectures']['resnet50']['total_parameters'] / 1e6:.1f} M",
        "File Size (MB)": bench_results["architectures"]["resnet50"]["model_file_size_mb"],
        "Mean Latency (ms)": resnet_lat["mean"],
        "P95 Latency (ms)": resnet_lat["p95"],
    }
])

display(bench_df)

# Visualization: Latency Comparison Bar Chart
fig, ax = plt.subplots(figsize=(7, 4))
models = ["MobileNetV3-Large", "ResNet50"]
latencies = [mobilenet_lat["mean"], resnet_lat["mean"]]
colors = ["#10b981", "#6366f1"]

bars = ax.bar(models, latencies, color=colors, width=0.5)
ax.set_ylabel("Mean Inference Latency (ms)")
ax.set_title("Single-Image CPU Inference Latency Comparison")
for bar in bars:
    yval = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2.0, yval + 1.0, f"{yval:.1f} ms", ha="center", va="bottom", fontweight="bold")
plt.tight_layout()
plt.show()

## 5. End-to-End Reproducibility Audit (`scripts/reproducibility_check.py`)
Audits Git status, DVC lockfile, manifests, configuration hashes, and hardware determinism boundaries.

In [ ]:
from scripts.reproducibility_check import run_reproducibility_audit

repro_res = run_reproducibility_audit()

print(f"\nOverall Reproducibility Passed: {repro_res['overall_reproducibility_passed']}")
for check_name, status_dict in repro_res["checks"].items():
    print(f"  - [{status_dict.get('status', 'OK')}] {check_name}")

print("\nHardware Determinism Boundary:")
print(repro_res["reproducibility_boundary"]["hardware_boundary_note"])

## 6. Documentation Citation Audit (`scripts/citation_audit.py`)
Audits all markdown documentation files to verify 100% official citation compliance.

In [ ]:
from scripts.citation_audit import run_citation_audit

cit_res = run_citation_audit()
print(f"Documentation Audit Passed: {cit_res['audit_passed']}")
print(f"Total Official Citations Verified: {cit_res['total_citations']}")

cit_md = (PROJECT_ROOT / "reports" / "citation_audit.md").read_text(encoding="utf-8")
print("\n" + cit_md[:600] + "\n...")

## 7. Interactive Live Screening & Grad-CAM Visualizer (`app/predictor.py`)
Demonstrates live inference on worker crop samples with prediction class, confidence, calibrated risk tier, recommendation, and Grad-CAM saliency overlay.

In [ ]:
from app.predictor import get_predictor
from app.risk import evaluate_risk_and_recommendation

predictor = get_predictor()
crops_dir = PROJECT_ROOT / "data" / "interim" / "crops"
sample_crops = sorted(list(crops_dir.glob("*.jpg")))[:3]

fig, axes = plt.subplots(len(sample_crops), 2, figsize=(10, 4 * len(sample_crops)))
if len(sample_crops) == 1:
    axes = [axes]

for idx, crop_path in enumerate(sample_crops):
    with Image.open(crop_path) as img:
        sample_img = img.convert("RGB")
    
    res, overlay = predictor.predict_with_gradcam(sample_img)
    risk_level, rec = evaluate_risk_and_recommendation(res["prediction"], res["confidence"])
    
    # Left: Raw Image
    axes[idx][0].imshow(sample_img)
    axes[idx][0].set_title(f"Raw Crop: {crop_path.stem}")
    axes[idx][0].axis("off")
    
    # Right: Grad-CAM Overlay
    axes[idx][1].imshow(overlay)
    axes[idx][1].set_title(f"Pred: {res['prediction']} ({res['confidence']*100:.1f}%) | Risk: {risk_level}")
    axes[idx][1].axis("off")
    
    print(f"Sample {crop_path.stem}:")
    print(f"  - Prediction: {res['prediction']} (Confidence: {res['confidence']*100:.1f}%)")
    print(f"  - Risk Level: {risk_level}")
    print(f"  - Recommendation: {rec}")
    print(f"  - Latency: {res['inference_latency_ms']:.2f} ms\n")

plt.tight_layout()
plt.show()